> <p><small>This notebook is made available subject to the licence and terms set out in <a href="https://creativecommons.org/licenses/by/4.0">https://creativecommons.org/licenses/by/4.0</a>.</small></p>

<img src="https://pub-bba109a9a6ac49e3b428cdca19c34363.r2.dev/LT%20-%20Session%202.jpg">

# 2.6 AI Long Activity 1 – Mini-Hackathon: Build a Mini‑Tokenizer (Teacher)

This notebook contains one possible solution for the mini‑tokenizer challenge lab; it is intended for teachers or for checking your work after attempting the student version.

60 minutes

## 1. Setup – Tiny Corpus

In [ ]:
corpus = [
    "low",
    "slow",
    "lotus",
    "follow",
    "yellow",
]

print("Corpus:")

for w in corpus:
    print("-", w)

Corpus:
- low
- slow
- lotus
- follow
- yellow


## 2. Represent words as character sequences

First convert each word into a list of characters.

In [ ]:
def word_to_chars(word: str):
    """Return a list of characters for the given word."""
    return list(word)


corpus_chars = [word_to_chars(w) for w in corpus]

corpus_chars

[['l', 'o', 'w'],
 ['s', 'l', 'o', 'w'],
 ['l', 'o', 't', 'u', 's'],
 ['f', 'o', 'l', 'l', 'o', 'w'],
 ['y', 'e', 'l', 'l', 'o', 'w']]

## 3. Count adjacent character pairs

Count how often each adjacent pair of symbols appears.

In [ ]:
from collections import Counter


def count_pairs(corpus_chars):
    """Count adjacent character pairs in the corpus."""
    pairs = Counter()

    for chars in corpus_chars:
        for i in range(len(chars) - 1):
            pair = (chars[i], chars[i + 1])
            pairs[pair] += 1

    return pairs


pair_counts = count_pairs(corpus_chars)

pair_counts

Counter({('l', 'o'): 5,
         ('o', 'w'): 4,
         ('l', 'l'): 2,
         ('s', 'l'): 1,
         ('o', 't'): 1,
         ('t', 'u'): 1,
         ('u', 's'): 1,
         ('f', 'o'): 1,
         ('o', 'l'): 1,
         ('y', 'e'): 1,
         ('e', 'l'): 1})

## 4. Apply one BPE‑style merge

Find the most frequent pair and merge it into a new symbol.

In [ ]:
def get_most_frequent_pair(pair_counts):
    """Return the most frequent adjacent pair."""
    return max(pair_counts.items(), key=lambda x: x[1])[0]


def merge_pair(corpus_chars, pair):
    """Merge all occurrences of a character pair in the corpus."""
    merged_corpus = []
    a, b = pair
    merged_symbol = a + b

    for chars in corpus_chars:
        new_chars = []
        i = 0

        while i < len(chars):
            if (
                i < len(chars) - 1
                and chars[i] == a
                and chars[i + 1] == b
            ):
                new_chars.append(merged_symbol)
                i += 2
            else:
                new_chars.append(chars[i])
                i += 1

        merged_corpus.append(new_chars)

    return merged_corpus


most_freq_pair = get_most_frequent_pair(pair_counts)

print(
    "Most frequent pair:",
    most_freq_pair,
)

corpus_chars_merged1 = merge_pair(
    corpus_chars,
    most_freq_pair,
)

corpus_chars_merged1

Most frequent pair: ('l', 'o')


[['lo', 'w'],
 ['s', 'lo', 'w'],
 ['lo', 't', 'u', 's'],
 ['f', 'o', 'l', 'lo', 'w'],
 ['y', 'e', 'l', 'lo', 'w']]

## 5. Repeat the merge step

Recompute pair counts and perform a second merge.

In [ ]:
pair_counts_2 = count_pairs(corpus_chars_merged1)
most_freq_pair_2 = get_most_frequent_pair(pair_counts_2)

print(
    "Second most frequent pair:",
    most_freq_pair_2,
)

corpus_chars_merged2 = merge_pair(
    corpus_chars_merged1,
    most_freq_pair_2,
)

corpus_chars_merged2

Second most frequent pair: ('lo', 'w')


[['low'],
 ['s', 'low'],
 ['lo', 't', 'u', 's'],
 ['f', 'o', 'l', 'low'],
 ['y', 'e', 'l', 'low']]

## 6. Build a vocabulary and token IDs

Build a vocabulary from the final merged corpus and map each symbol to an integer ID.

In [ ]:
def build_vocab(corpus_chars):
    """Build a sorted vocabulary from the corpus."""
    symbols = set()

    for chars in corpus_chars:
        for c in chars:
            symbols.add(c)

    return sorted(symbols)


vocab = build_vocab(corpus_chars_merged2)
token_to_id = {
    tok: i for i, tok in enumerate(vocab)
}

token_to_id

{'e': 0,
 'f': 1,
 'l': 2,
 'lo': 3,
 'low': 4,
 'o': 5,
 's': 6,
 't': 7,
 'u': 8,
 'y': 9}

## 7. Convert text to token IDs

Reuse the same merge operations and apply them to new words before mapping to IDs.

In [ ]:
def encode_word(word, merges, token_to_id):
    """Encode a word using the learned merges and token mapping."""
    chars = list(word)

    for pair in merges:
        chars = merge_pair([chars], pair)[0]

    return [
        token_to_id[c]
        for c in chars
        if c in token_to_id
    ]


merges = [
    most_freq_pair,
    most_freq_pair_2,
]

print(
    "Encoding 'low':",
    encode_word("low", merges, token_to_id),
)
print(
    "Encoding 'yellow':",
    encode_word("yellow", merges, token_to_id),
)
print(
    "Encoding 'follow':",
    encode_word("follow", merges, token_to_id),
)

Encoding 'low': [4]
Encoding 'yellow': [9, 0, 2, 4]
Encoding 'follow': [1, 5, 2, 4]


> 💭 **Reflection (for teachers)**
>
> Key teaching points:
>
> - Students see that even a tiny BPE‑style algorithm already changes the units the model sees.
> - Different corpora would lead to different merges and vocabularies.
> - This motivates why tokenization choices are design decisions with real consequences, especially for under‑resourced languages.
